# Phase 1: AutoPET-I (FDG) — Extract Lesion Features

Processes AutoPET-I FDAT data directly from the zip on Google Drive.
Extracts only SUV.nii.gz + SEG.nii.gz per case to /tmp, processes,
deletes. Colab disk usage stays under 500 MB.

Fully resumable across sessions.

**Pre-registration:** https://doi.org/10.17605/OSF.IO/4KAZN

In [ ]:
# Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install -q nibabel

In [ ]:
# Step 2: Inspect zip structure
import zipfile
import os

zip_path = '/content/drive/MyDrive/P79 Data/autopet_i/fdg-pet-ct-lesions.zip'
print(f'Zip size: {os.path.getsize(zip_path)/(1024**3):.1f} GB')

with zipfile.ZipFile(zip_path, 'r') as zf:
    all_names = zf.namelist()
    niftis = [n for n in all_names if n.endswith('.nii.gz')]

print(f'Total entries: {len(all_names)}')
print(f'NIfTI files: {len(niftis)}')
print(f'Expected cases: {len(niftis) // 5}')

# Group by case
cases = {}
for name in all_names:
    if not name.endswith('.nii.gz'):
        continue
    parts = name.split('/')
    if len(parts) >= 3:
        case_id = parts[1]
        filename = parts[-1]
        if case_id not in cases:
            cases[case_id] = {}
        cases[case_id][filename] = name

# Verify all cases have SUV and SEG
has_suv = sum(1 for c in cases.values() if 'SUV.nii.gz' in c)
has_seg = sum(1 for c in cases.values() if 'SEG.nii.gz' in c)
has_both = sum(1 for c in cases.values() if 'SUV.nii.gz' in c and 'SEG.nii.gz' in c)
print(f'\nCases: {len(cases)}')
print(f'With SUV: {has_suv}')
print(f'With SEG: {has_seg}')
print(f'With both: {has_both}')

# Show files per case for first example
first_case = sorted(cases.keys())[0]
print(f'\nExample case {first_case}:')
for fname, full_path in cases[first_case].items():
    print(f'  {fname}')

In [ ]:
# Step 3: Test one case — verify SUV values look correct
import nibabel as nib
import numpy as np
import shutil

first_case = sorted(cases.keys())[0]
print(f'Testing case: {first_case}')

# Extract SUV and SEG to /tmp
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extract(cases[first_case]['SUV.nii.gz'], '/tmp/test_case')
    zf.extract(cases[first_case]['SEG.nii.gz'], '/tmp/test_case')

suv_local = f'/tmp/test_case/{cases[first_case]["SUV.nii.gz"]}'
seg_local = f'/tmp/test_case/{cases[first_case]["SEG.nii.gz"]}'

suv_img = nib.load(suv_local)
suv = suv_img.get_fdata()
spacing = suv_img.header.get_zooms()[:3]

seg_img = nib.load(seg_local)
seg = seg_img.get_fdata()

print(f'SUV shape: {suv.shape}')
print(f'SEG shape: {seg.shape}')
print(f'Voxel spacing (mm): {spacing}')
print(f'SUV range: [{suv.min():.2f}, {suv.max():.2f}]')
print(f'SUV mean (non-zero): {suv[suv > 0].mean():.2f}')
print(f'SEG unique values: {np.unique(seg)}')
print(f'SEG positive voxels: {(seg > 0).sum()}')

if suv.max() < 100 and suv.min() >= -0.1:
    print('\nSUV values look correct (typical FDG range)')
else:
    print(f'\nWARNING: SUV range looks unusual')

# Quick lesion check
from scipy import ndimage
binary = (seg > 0).astype(np.int32)
labelled, n_lesions = ndimage.label(binary)
voxel_vol_ml = float(np.prod(spacing)) / 1000.0
print(f'\nLesions (connected components): {n_lesions}')
for i in range(1, min(n_lesions + 1, 6)):
    mask = labelled == i
    vol = mask.sum() * voxel_vol_ml
    s = suv[mask]
    print(f'  Lesion {i}: {vol:.1f} mL, SUVmax={s.max():.1f}, SUVmean={s.mean():.1f}')

shutil.rmtree('/tmp/test_case', ignore_errors=True)

In [ ]:
# Step 4: Extract lesion features for ALL cases
# Processes directly from zip: extract 2 files to /tmp, process, delete.
# Colab disk stays under 500 MB. Fully resumable.

import pandas as pd
import time

MIN_VOLUME_ML = 1.0  # pre-registered §3.2
output_path = '/content/drive/MyDrive/P79 Data/autopet_i/lesion_features.parquet'
progress_path = '/content/drive/MyDrive/P79 Data/autopet_i/_processed_cases.txt'

# Resume support
processed = set()
if os.path.exists(progress_path):
    with open(progress_path) as f:
        processed = set(line.strip() for line in f if line.strip())
    print(f'Resuming: {len(processed)} cases already processed')

# Load existing features if resuming
all_features = []
if os.path.exists(output_path) and len(processed) > 0:
    existing_df = pd.read_parquet(output_path)
    all_features = existing_df.to_dict('records')
    print(f'Loaded {len(all_features)} existing lesion features')

# Filter to processable cases
valid_cases = {k: v for k, v in cases.items()
               if 'SUV.nii.gz' in v and 'SEG.nii.gz' in v}
to_process = {k: v for k, v in valid_cases.items() if k not in processed}

print(f'Valid cases: {len(valid_cases)}')
print(f'To process: {len(to_process)}')
print(f'Starting...\n')

start_time = time.time()
new_count = 0
errors = []

for case_id, files in sorted(to_process.items()):
    try:
        # Extract SUV + SEG to /tmp
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extract(files['SUV.nii.gz'], '/tmp/current_case')
            zf.extract(files['SEG.nii.gz'], '/tmp/current_case')

        suv_local = f'/tmp/current_case/{files["SUV.nii.gz"]}'
        seg_local = f'/tmp/current_case/{files["SEG.nii.gz"]}'

        # Load
        suv_img = nib.load(suv_local)
        suv = suv_img.get_fdata().astype(np.float64)
        spacing = suv_img.header.get_zooms()[:3]

        seg_img = nib.load(seg_local)
        seg = seg_img.get_fdata().astype(np.int32)

        if suv.shape != seg.shape:
            errors.append((case_id, f'Shape mismatch: SUV {suv.shape} vs SEG {seg.shape}'))
            shutil.rmtree('/tmp/current_case', ignore_errors=True)
            continue

        # Connected components
        voxel_vol_ml = float(np.prod(spacing)) / 1000.0
        binary = (seg > 0).astype(np.int32)
        labelled, n_comp = ndimage.label(binary)

        for comp_id in range(1, n_comp + 1):
            comp_mask = labelled == comp_id
            n_vox = int(comp_mask.sum())
            vol_ml = n_vox * voxel_vol_ml

            if vol_ml < MIN_VOLUME_ML:
                continue

            suv_lesion = suv[comp_mask]
            coords = np.argwhere(comp_mask)
            centroid = coords.mean(axis=0)

            all_features.append({
                'case_id': case_id,
                'lesion_id': comp_id,
                'suvmax': float(suv_lesion.max()),
                'suvmean': float(suv_lesion.mean()),
                'tlg': float(suv_lesion.mean()) * vol_ml,
                'volume_ml': vol_ml,
                'n_voxels': n_vox,
                'centroid_0': float(centroid[0]),
                'centroid_1': float(centroid[1]),
                'centroid_2': float(centroid[2]),
                'voxel_spacing_0': float(spacing[0]),
                'voxel_spacing_1': float(spacing[1]),
                'voxel_spacing_2': float(spacing[2]),
                'dataset': 'autopet_i',
                'tracer': 'FDG',
                'vendor': 'Siemens',
            })

        # Clean up
        shutil.rmtree('/tmp/current_case', ignore_errors=True)

        # Log progress
        with open(progress_path, 'a') as f:
            f.write(case_id + '\n')
        processed.add(case_id)
        new_count += 1

    except Exception as e:
        errors.append((case_id, str(e)))
        shutil.rmtree('/tmp/current_case', ignore_errors=True)

    # Progress every 25 cases
    if new_count > 0 and new_count % 25 == 0:
        elapsed = time.time() - start_time
        rate = new_count / elapsed * 3600
        remaining = (len(to_process) - new_count) / rate if rate > 0 else 0
        print(f'  {len(processed)}/{len(valid_cases)} cases | '
              f'{len(all_features)} lesions | '
              f'{len(errors)} errors | '
              f'{rate:.0f} cases/hr | '
              f'~{remaining:.1f}h remaining')

        # Incremental save
        pd.DataFrame(all_features).to_parquet(output_path, index=False)

# Final save
if all_features:
    df = pd.DataFrame(all_features)
    df.to_parquet(output_path, index=False)
    elapsed = time.time() - start_time
    print(f'\nDone: {len(df)} lesions from {len(processed)} cases in {elapsed/3600:.1f}h')
    print(f'Saved to {output_path}')
else:
    print('No features extracted')

if errors:
    print(f'\n{len(errors)} errors:')
    for c, e in errors[:10]:
        print(f'  {c}: {e}')

In [ ]:
# Step 5: Summary of extracted features
output_path = '/content/drive/MyDrive/P79 Data/autopet_i/lesion_features.parquet'

if os.path.exists(output_path):
    df = pd.read_parquet(output_path)
    print(f'Total lesions (>=1 mL): {len(df)}')
    print(f'Total cases with lesions: {df["case_id"].nunique()}')
    print(f'Lesions per case: mean={df.groupby("case_id").size().mean():.1f}, '
          f'median={df.groupby("case_id").size().median():.0f}, '
          f'max={df.groupby("case_id").size().max()}')
    print(f'\nVolume (mL):')
    print(f'  median={df["volume_ml"].median():.1f}, '
          f'IQR=[{df["volume_ml"].quantile(0.25):.1f}, {df["volume_ml"].quantile(0.75):.1f}], '
          f'range=[{df["volume_ml"].min():.1f}, {df["volume_ml"].max():.1f}]')
    print(f'\nSUVmax:')
    print(f'  median={df["suvmax"].median():.1f}, '
          f'IQR=[{df["suvmax"].quantile(0.25):.1f}, {df["suvmax"].quantile(0.75):.1f}], '
          f'range=[{df["suvmax"].min():.1f}, {df["suvmax"].max():.1f}]')
    print(f'\nSUVmean:')
    print(f'  median={df["suvmean"].median():.1f}, '
          f'IQR=[{df["suvmean"].quantile(0.25):.1f}, {df["suvmean"].quantile(0.75):.1f}]')
    print(f'\nTLG:')
    print(f'  median={df["tlg"].median():.1f}, '
          f'range=[{df["tlg"].min():.1f}, {df["tlg"].max():.1f}]')
    print(f'\nVolume quartile distribution:')
    df['vol_q'] = pd.qcut(df['volume_ml'], 4, labels=['Q1','Q2','Q3','Q4'])
    for q in ['Q1','Q2','Q3','Q4']:
        sub = df[df['vol_q'] == q]
        print(f'  {q}: n={len(sub)}, vol range=[{sub["volume_ml"].min():.1f}, {sub["volume_ml"].max():.1f}] mL, '
              f'median SUVmax={sub["suvmax"].median():.1f}')
    print(f'\nSUVmax > 50 (flagged per §3.9): {(df["suvmax"] > 50).sum()}')
    print(f'SUVmax < 0 (should be excluded): {(df["suvmax"] < 0).sum()}')
else:
    print('No feature table found — run Step 4 first')

In [ ]:
# Step 6: Colab disk check — should be nearly empty
import shutil
total, used, free = shutil.disk_usage('/content')
print(f'Colab disk: {used/(1024**3):.1f} GB used / {total/(1024**3):.1f} GB total')

In [ ]:
# Step 7: Triage SUVmax > 50 outliers per §3.9
# Categorises each flagged lesion by likely cause to prioritise manual image review.
# Categories:
#   A_extreme_suv       SUVmax > 150  — physically rare; almost always artifact/contamination
#   B_small_high_suv    vol < 2 mL    — small flagged lesions = typical bladder/brain/injection bleed-in
#   C_focal_hotspot     SUVmean/SUVmax < 0.30 — focal hot voxel in cool mask (reconstruction artifact)
#   D_likely_real       remainder      — 50<SUVmax<=150 with reasonable uniformity in larger lesion (auto-retain)
output_path = '/content/drive/MyDrive/P79 Data/autopet_i/lesion_features.parquet'
review_path = '/content/drive/MyDrive/P79 Data/autopet_i/suv_outlier_review.csv'

df = pd.read_parquet(output_path)
flagged = df[df['suvmax'] > 50].copy()

flagged['centroid_z_mm'] = flagged['centroid_2'] * flagged['voxel_spacing_2']
flagged['suv_uniformity'] = flagged['suvmean'] / flagged['suvmax']

def categorise(row):
    if row['suvmax'] > 150:
        return 'A_extreme_suv'
    if row['volume_ml'] < 2.0:
        return 'B_small_high_suv'
    if row['suv_uniformity'] < 0.30:
        return 'C_focal_hotspot'
    return 'D_likely_real'

flagged['triage_category'] = flagged.apply(categorise, axis=1)
flagged['needs_image_review'] = flagged['triage_category'] != 'D_likely_real'

flagged = flagged.sort_values(['triage_category', 'suvmax'], ascending=[True, False])

out_cols = [
    'case_id', 'lesion_id', 'triage_category', 'needs_image_review',
    'suvmax', 'suvmean', 'suv_uniformity', 'volume_ml',
    'centroid_2', 'centroid_z_mm', 'voxel_spacing_2',
]
flagged[out_cols].to_csv(review_path, index=False)

print(f'Total SUVmax > 50 lesions flagged: {len(flagged)}')
print(f'Unique cases affected: {flagged["case_id"].nunique()}')
print(f'\nTriage breakdown:')
for cat, n in flagged['triage_category'].value_counts().sort_index().items():
    sub = flagged[flagged['triage_category'] == cat]
    print(f'  {cat}: n={n}, '
          f'SUVmax median={sub["suvmax"].median():.1f}, '
          f'volume median={sub["volume_ml"].median():.2f} mL')

needs_n = int(flagged['needs_image_review'].sum())
needs_cases = flagged[flagged['needs_image_review']]['case_id'].nunique()
auto_n = len(flagged) - needs_n
print(f'\nNeeds image review: {needs_n} lesions across {needs_cases} cases')
print(f'Auto-retain (likely real disease): {auto_n} lesions')

print(f'\nTop 15 by SUVmax:')
print(flagged[['case_id','lesion_id','triage_category','suvmax','suvmean','suv_uniformity','volume_ml']].head(15).to_string(index=False))

print(f'\nReview CSV saved to: {review_path}')

In [ ]:
# Step 8: Per-case concentration diagnostic for flagged outliers
# Determines whether the 158 review-needed lesions concentrate in 2-3 cases
# (case-level mechanism: scanner setting, segmentation quirk, single high-burden patient)
# or spread across all 18 cases (population-level biology / artifact pattern).

review_path = '/content/drive/MyDrive/P79 Data/autopet_i/suv_outlier_review.csv'
output_path = '/content/drive/MyDrive/P79 Data/autopet_i/lesion_features.parquet'

flagged = pd.read_csv(review_path)
df = pd.read_parquet(output_path)

per_case = flagged.groupby('case_id').agg(
    flagged_n=('lesion_id', 'count'),
    cat_A=('triage_category', lambda s: (s == 'A_extreme_suv').sum()),
    cat_B=('triage_category', lambda s: (s == 'B_small_high_suv').sum()),
    cat_C=('triage_category', lambda s: (s == 'C_focal_hotspot').sum()),
    cat_D=('triage_category', lambda s: (s == 'D_likely_real').sum()),
    suvmax_max=('suvmax', 'max'),
    suvmax_med=('suvmax', 'median'),
).join(df.groupby('case_id').size().rename('total_lesions_in_case'), how='left')
per_case['flagged_pct'] = (per_case['flagged_n'] / per_case['total_lesions_in_case'] * 100).round(1)
per_case = per_case.sort_values('flagged_n', ascending=False)

print('Per-case breakdown of flagged lesions (sorted by flag count):')
print(per_case.to_string())

print(f'\nCumulative concentration of Category C (n=120):')
c_only = per_case[per_case['cat_C'] > 0].sort_values('cat_C', ascending=False)
total_c = int(c_only['cat_C'].sum())
cum = 0
for i, (case, row) in enumerate(c_only.iterrows(), 1):
    cum += int(row['cat_C'])
    print(f'  Top {i} ({case}): +{int(row["cat_C"])}  cumulative {cum}/{total_c} ({cum/total_c*100:.0f}%)')
    if cum / total_c >= 0.90:
        print(f'  ...top {i} cases account for >=90% of Category C')
        break

print(f'\nInterpretation:')
n_top_cases_50pct = (c_only['cat_C'].cumsum() / total_c >= 0.50).idxmax()
top_cases_50 = c_only.index.get_loc(n_top_cases_50pct) + 1
if top_cases_50 <= 3:
    print(f'  Category C is CASE-CONCENTRATED ({top_cases_50} cases hold >=50% of C lesions).')
    print(f'  Mechanism is likely case-level (one scanner session, one segmentation quirk,')
    print(f'  one high-burden patient). Review at the case level, not lesion level.')
else:
    print(f'  Category C is DISTRIBUTED ({top_cases_50} cases hold >=50% of C lesions).')
    print(f'  Mechanism is more likely systematic (biology of necrotic-centred tumours,')
    print(f'  or scanner/protocol effect across multiple patients). Image review needed per case.')

In [ ]:
# Step 9: Augment lesion_features.parquet with SUVpeak (per pre-reg §3.3 / project plan)
#
# SUVpeak = mean SUV inside a 1 cm³ sphere centred on the lesion's maximum-uptake voxel.
# This is the QIBA / PERCIST-recommended robust analogue of SUVmax. Our pre-registration
# specifies SUVmax + SUVpeak + SUVmean; the original Step 4 only extracted SUVmax + SUVmean,
# so this step adds the missing column without disturbing the existing parquet.
#
# Method: re-extract SUV+SEG per case, redo connected components, match each parquet lesion
# to its component by centroid (1-voxel tolerance), find the SUVmax voxel within the mask,
# build a 1 cm³ sphere around it, average. Resumable via _suvpeak_processed.txt.
# DEVIATION from strict QIBA: sphere is centred on the max-uptake voxel rather than position-
# optimised. To be documented in osf/amendment_log.md before Freeze Gate 1.

input_path = '/content/drive/MyDrive/P79 Data/autopet_i/lesion_features.parquet'
output_path = '/content/drive/MyDrive/P79 Data/autopet_i/lesion_features_v2.parquet'
progress_path = '/content/drive/MyDrive/P79 Data/autopet_i/_suvpeak_processed.txt'

base_df = pd.read_parquet(input_path)
print(f'Loaded {len(base_df)} lesions from {base_df["case_id"].nunique()} cases')

processed = set()
if os.path.exists(progress_path):
    with open(progress_path) as f:
        processed = set(line.strip() for line in f if line.strip())
    print(f'Resuming: {len(processed)} cases already augmented')

augmented_dict = {}
if os.path.exists(output_path) and processed:
    prior = pd.read_parquet(output_path)
    for _, r in prior[['case_id','lesion_id','suvpeak']].dropna(subset=['suvpeak']).iterrows():
        augmented_dict[(r['case_id'], int(r['lesion_id']))] = float(r['suvpeak'])
    print(f'Loaded {len(augmented_dict)} prior SUVpeak values')

def compute_suvpeak(suv, spacing, max_voxel_idx, sphere_volume_ml=1.0):
    radius_mm = (3.0 * sphere_volume_ml * 1000.0 / (4.0 * np.pi)) ** (1.0/3.0)
    spacing_arr = np.array(spacing, dtype=np.float64)
    half_vox = np.ceil(radius_mm / spacing_arr).astype(int)
    cz, cy, cx = max_voxel_idx
    z0, z1 = max(cz - half_vox[0], 0), min(cz + half_vox[0] + 1, suv.shape[0])
    y0, y1 = max(cy - half_vox[1], 0), min(cy + half_vox[1] + 1, suv.shape[1])
    x0, x1 = max(cx - half_vox[2], 0), min(cx + half_vox[2] + 1, suv.shape[2])
    zz, yy, xx = np.meshgrid(
        np.arange(z0, z1) - cz,
        np.arange(y0, y1) - cy,
        np.arange(x0, x1) - cx,
        indexing='ij',
    )
    dist_mm = np.sqrt((zz * spacing_arr[0])**2
                      + (yy * spacing_arr[1])**2
                      + (xx * spacing_arr[2])**2)
    sphere_mask = dist_mm <= radius_mm
    return float(suv[z0:z1, y0:y1, x0:x1][sphere_mask].mean())

all_cases = sorted(set(base_df['case_id'].unique()))
to_process = [c for c in all_cases if c not in processed]
print(f'Cases to augment: {len(to_process)}')

start_time = time.time()
new_count = 0
errors = []

for case_id in to_process:
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extract(cases[case_id]['SUV.nii.gz'], '/tmp/aug_case')
            zf.extract(cases[case_id]['SEG.nii.gz'], '/tmp/aug_case')

        suv_local = f'/tmp/aug_case/{cases[case_id]["SUV.nii.gz"]}'
        seg_local = f'/tmp/aug_case/{cases[case_id]["SEG.nii.gz"]}'

        suv_img = nib.load(suv_local)
        suv = suv_img.get_fdata().astype(np.float64)
        spacing = suv_img.header.get_zooms()[:3]
        seg = nib.load(seg_local).get_fdata().astype(np.int32)

        if suv.shape != seg.shape:
            errors.append((case_id, f'shape mismatch'))
            shutil.rmtree('/tmp/aug_case', ignore_errors=True)
            continue

        binary = (seg > 0).astype(np.int32)
        labelled, n_comp = ndimage.label(binary)

        # Vectorised centroid + size for all components
        comp_ids = list(range(1, n_comp + 1))
        comp_centroids = np.array(ndimage.center_of_mass(binary, labelled, comp_ids))
        comp_sizes = np.array(ndimage.sum(binary, labelled, comp_ids))
        voxel_vol_ml = float(np.prod(spacing)) / 1000.0
        comp_volumes = comp_sizes * voxel_vol_ml

        # Filter to ≥1 mL components (matches Step 4 inclusion rule)
        valid = comp_volumes >= 1.0
        valid_ids = np.array(comp_ids)[valid]
        valid_centroids = comp_centroids[valid]

        case_lesions = base_df[base_df['case_id'] == case_id]

        for _, row in case_lesions.iterrows():
            target = np.array([row['centroid_0'], row['centroid_1'], row['centroid_2']])
            dists = np.linalg.norm(valid_centroids - target, axis=1)
            best = int(np.argmin(dists))
            if dists[best] > 1.0:
                errors.append((case_id, f'lesion {row["lesion_id"]}: centroid match dist={dists[best]:.2f} vox'))
                continue
            comp_id = int(valid_ids[best])
            comp_mask = labelled == comp_id
            suv_in_lesion = np.where(comp_mask, suv, -np.inf)
            max_idx = np.unravel_index(np.argmax(suv_in_lesion), suv.shape)
            sp = compute_suvpeak(suv, spacing, max_idx, sphere_volume_ml=1.0)
            augmented_dict[(case_id, int(row['lesion_id']))] = sp

        shutil.rmtree('/tmp/aug_case', ignore_errors=True)
        with open(progress_path, 'a') as f:
            f.write(case_id + '\n')
        processed.add(case_id)
        new_count += 1

    except Exception as e:
        errors.append((case_id, str(e)))
        shutil.rmtree('/tmp/aug_case', ignore_errors=True)

    if new_count > 0 and new_count % 25 == 0:
        elapsed = time.time() - start_time
        rate = new_count / elapsed * 3600
        remaining = (len(to_process) - new_count) / rate if rate > 0 else 0
        print(f'  {len(processed)}/{len(all_cases)} cases | '
              f'{len(augmented_dict)} suvpeak values | '
              f'{rate:.0f} cases/hr | ~{remaining:.1f}h remaining')
        # Incremental save
        sp_idx = pd.MultiIndex.from_tuples(list(augmented_dict.keys()), names=['case_id','lesion_id'])
        sp_series = pd.Series(list(augmented_dict.values()), index=sp_idx, name='suvpeak').reset_index()
        base_df.merge(sp_series, on=['case_id','lesion_id'], how='left').to_parquet(output_path, index=False)

# Final merge + save
sp_idx = pd.MultiIndex.from_tuples(list(augmented_dict.keys()), names=['case_id','lesion_id'])
sp_series = pd.Series(list(augmented_dict.values()), index=sp_idx, name='suvpeak').reset_index()
merged = base_df.merge(sp_series, on=['case_id','lesion_id'], how='left')
merged.to_parquet(output_path, index=False)
elapsed = time.time() - start_time

n_with_sp = int((~merged['suvpeak'].isna()).sum())
n_missing = int(merged['suvpeak'].isna().sum())
print(f'\nDone in {elapsed/3600:.1f}h')
print(f'Lesions with SUVpeak: {n_with_sp}/{len(merged)}, missing: {n_missing}')
print(f'Saved to {output_path}')

if n_with_sp > 0:
    ratio = merged['suvpeak'] / merged['suvmax']
    print(f'\nSUVpeak distribution:')
    print(f'  median={merged["suvpeak"].median():.1f}, '
          f'IQR=[{merged["suvpeak"].quantile(0.25):.1f}, {merged["suvpeak"].quantile(0.75):.1f}]')
    print(f'\nSUVpeak / SUVmax ratio (typical 0.4-0.9; <0.3 = SUVmax dominated by single voxel):')
    print(f'  median={ratio.median():.3f}, mean={ratio.mean():.3f}')
    print(f'  ratio < 0.5: {int((ratio < 0.5).sum())} lesions  (single-voxel dominated)')
    print(f'  ratio < 0.3: {int((ratio < 0.3).sum())} lesions  (severe single-voxel artifact)')
    print(f'\nFor §3.9 outliers (SUVmax > 50): SUVpeak distribution by triage category')
    flagged_merged = merged[merged['suvmax'] > 50].copy()
    if len(flagged_merged) > 0:
        review = pd.read_csv('/content/drive/MyDrive/P79 Data/autopet_i/suv_outlier_review.csv')
        flagged_merged = flagged_merged.merge(review[['case_id','lesion_id','triage_category']], on=['case_id','lesion_id'], how='left')
        for cat in sorted(flagged_merged['triage_category'].dropna().unique()):
            sub = flagged_merged[flagged_merged['triage_category'] == cat]
            print(f'  {cat}: n={len(sub)}, '
                  f'SUVmax median={sub["suvmax"].median():.1f}, '
                  f'SUVpeak median={sub["suvpeak"].median():.1f}, '
                  f'ratio median={(sub["suvpeak"]/sub["suvmax"]).median():.3f}')

if errors:
    print(f'\n{len(errors)} errors:')
    for c, e in errors[:10]:
        print(f'  {c}: {e}')